<a href="https://www.kaggle.com/code/shamanthakreddymallu/f1-2022-data-script?scriptVersionId=325911321" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Imports

In [1]:
!pip install fastf1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00


In [2]:
import fastf1
import pandas as pd
import os
import shutil
from datetime import datetime

# Configuration

In [3]:
#schedule = fastf1.get_event_schedule(2017, include_testing=False)
#print(schedule[['RoundNumber', 'EventName', 'EventFormat', 'Session1', 'Session2', 'Session3', 'Session4', 'Session5']].to_string())

In [4]:
YEAR        = 2022
START_ROUND = 1     # from this round
END_ROUND   = None    # stop after this round (set to None for full season)
ERA_DIR     = 'f1-ground-effect-hybrid-era'
CACHE_DIR   = './fastf1_cache'

os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

# Helper Functions

In [5]:
def td_to_sec(x):
    return round(x.total_seconds(), 3) if pd.notnull(x) else None


def fmt_race_time(x):
    if pd.isnull(x):
        return None
    total = x.total_seconds()
    h = int(total // 3600)
    m = int((total % 3600) // 60)
    s = total % 60
    return f"{h}:{m:02d}:{s:06.3f}" if h else f"{m}:{s:06.3f}"


def save_results(session, path):
    """Save session classification results."""
    cols_available = [c for c in [
        'DriverNumber', 'Abbreviation', 'FullName', 'TeamName',
        'GridPosition', 'Position', 'Points', 'Status', 'Time', 'Q1', 'Q2', 'Q3'
    ] if c in session.results.columns]

    results = session.results[cols_available].copy()
    rename = {
        'DriverNumber': 'driver_number', 'Abbreviation': 'abbreviation',
        'FullName': 'full_name', 'TeamName': 'team',
        'GridPosition': 'grid', 'Position': 'position',
        'Points': 'points', 'Status': 'status',
        'Time': 'time', 'Q1': 'q1', 'Q2': 'q2', 'Q3': 'q3'
    }
    results = results.rename(columns={k: v for k, v in rename.items() if k in results.columns})

    for col in ['time', 'q1', 'q2', 'q3']:
        if col in results.columns:
            results[col] = results[col].apply(fmt_race_time)

    results.to_csv(path, index=False)
    print(f"      results: {len(results)} drivers")


def save_laps(session, path):
    """Save lap times + telemetry for any session type."""
    lap_cols = [c for c in [
        'Driver', 'Team', 'LapNumber', 'LapTime',
        'Sector1Time', 'Sector2Time', 'Sector3Time',
        'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST',
        'Compound', 'TyreLife', 'FreshTyre', 'Stint',
        'PitInTime', 'PitOutTime', 'Position',
        'IsPersonalBest', 'TrackStatus'
    ] if c in session.laps.columns]

    laps = session.laps[lap_cols].copy()
    rename = {
        'Driver': 'driver', 'Team': 'team', 'LapNumber': 'lap',
        'LapTime': 'lap_time', 'Sector1Time': 's1', 'Sector2Time': 's2',
        'Sector3Time': 's3', 'SpeedI1': 'speed_i1', 'SpeedI2': 'speed_i2',
        'SpeedFL': 'speed_fl', 'SpeedST': 'speed_st', 'Compound': 'compound',
        'TyreLife': 'tyre_life', 'FreshTyre': 'fresh_tyre', 'Stint': 'stint',
        'PitInTime': 'pit_in', 'PitOutTime': 'pit_out', 'Position': 'position',
        'IsPersonalBest': 'is_personal_best', 'TrackStatus': 'track_status'
    }
    laps = laps.rename(columns={k: v for k, v in rename.items() if k in laps.columns})

    for col in ['lap_time', 's1', 's2', 's3', 'pit_in', 'pit_out']:
        if col in laps.columns:
            laps[col] = laps[col].apply(td_to_sec)

    # car telemetry aggregates per lap
    records = []
    for _, lap in session.laps.iterlaps():
        row = {'driver': lap['Driver'], 'lap': lap['LapNumber']}
        try:
            car = lap.get_car_data()
            row['full_throttle_pct'] = round(float((car['Throttle'] == 100).mean() * 100), 2)
            row['brake_pct']         = round(float((car['Brake'] > 0).mean() * 100), 2)
            row['gear_changes']      = int((car['nGear'].diff().fillna(0) != 0).sum())
            row['max_speed']         = round(float(car['Speed'].max()), 1)
            row['min_speed']         = round(float(car['Speed'].min()), 1)
            row['drs_activated']     = bool((car['DRS'] >= 10).any())
        except Exception:
            row.update({
                'full_throttle_pct': None, 'brake_pct': None,
                'gear_changes': None, 'max_speed': None,
                'min_speed': None, 'drs_activated': None
            })
        records.append(row)

    car_df = pd.DataFrame(records)
    laps = laps.merge(car_df, on=['driver', 'lap'], how='left')
    laps.to_csv(path, index=False)
    print(f"      laps: {len(laps)} laps, {len(laps.columns)} columns")


def collect_session(year, gp_name, session_key, laps_path, results_path):
    """Load a session and save both laps and results."""
    try:
        session = fastf1.get_session(year, gp_name, session_key)
        session.load(telemetry=True, weather=False, messages=False)
        save_results(session, results_path)
        save_laps(session, laps_path)
    except Exception as e:
        print(f"      ERROR: {e}")

# Main Code

In [6]:
schedule = fastf1.get_event_schedule(YEAR, include_testing=False)
total = len(schedule)
print(f"\nF1 {YEAR} — {total} rounds found\n{'='*60}\n")

for _, event in schedule.iterrows():
    round_num = int(event['RoundNumber'])
    gp_name   = event['EventName']
    fmt       = event['EventFormat'].lower()
    is_sprint = 'sprint' in fmt
    gp_folder = f"{round_num:02d}_{gp_name.replace(' ', '_')}"
    OUT       = os.path.join(ERA_DIR, str(YEAR), gp_folder)
    os.makedirs(OUT, exist_ok=True)

    if round_num < START_ROUND:
        continue
    if END_ROUND is not None and round_num > END_ROUND:
        break

    print(f"[{round_num:02d}/{total}] {gp_name} {'(Sprint)' if is_sprint else ''}")

    if event['EventDate'].date() > datetime.today().date():
        print("    upcoming race, skipping.\n")
        continue

    # sessions differ by weekend format
    if is_sprint:
        sessions = [
            ('FP1',  'practice_1_laps.csv',          'practice_1_results.csv'),
            ('SQ',   'sprint_qualifying_laps.csv',    'sprint_qualifying_results.csv'),
            ('S',    'sprint_laps.csv',               'sprint_results.csv'),
            ('Q',    'qualifying_laps.csv',           'qualifying_results.csv'),
            ('R',    'race_laps.csv',                 'race_results.csv'),
        ]
    else:
        sessions = [
            ('FP1',  'practice_1_laps.csv',   'practice_1_results.csv'),
            ('FP2',  'practice_2_laps.csv',   'practice_2_results.csv'),
            ('FP3',  'practice_3_laps.csv',   'practice_3_results.csv'),
            ('Q',    'qualifying_laps.csv',   'qualifying_results.csv'),
            ('R',    'race_laps.csv',          'race_results.csv'),
        ]

    for session_key, laps_file, results_file in sessions:
        print(f"    [{session_key}] {laps_file.replace('_laps.csv','').replace('_',' ')}")
        collect_session(
            YEAR, gp_name, session_key,
            os.path.join(OUT, laps_file),
            os.path.join(OUT, results_file)
        )

    print()

core           INFO 	Loading data for Bahrain Grand Prix - Practice 1 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...



F1 2022 — 22 rounds found

[01/22] Bahrain Grand Prix 
    [FP1] practice 1


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to

      results: 20 drivers


core           INFO 	Loading data for Bahrain Grand Prix - Practice 2 [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


...

      ERROR: any API: 500 calls/h
    [FP3] practice 3


core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


      ERROR: any API: 500 calls/h
    [Q] qualifying


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


      ERROR: any API: 500 calls/h
    [R] race
      ERROR: any API: 500 calls/h



# Final Checks

In [7]:
# cleanup cache
if os.path.exists(CACHE_DIR):
    shutil.rmtree(CACHE_DIR)
    print("Cache cleared.")

print(f"\nSeason download complete! Data saved under: {ERA_DIR}/{YEAR}/")

Cache cleared.

Season download complete! Data saved under: f1-ground-effect-hybrid-era/2022/
